# 02 - Cleaning, normalisation and the inclusion funnel

Input: `data/raw/reviews_dataset.csv` (the scraper output, never edited by hand).

This notebook recomputes every derived column from the raw text and star rating using the rules in `src/preprocessing.py`, then applies the pre-registered inclusion criteria in order:

1. remove empty review text
2. remove exact duplicates (same `review_id` or same normalised text)
3. keep substantive reviews (>= 6 words)
4. drop the neutral band (3 stars) and reviews with no star rating
5. restrict to reviews whose dominant script is Arabic or English

Row counts before and after each step are written to `data/interim/cleaning_funnel.csv` and are the numbers quoted in the Methods chapter.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make src importable when the kernel starts inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")
from src import config
from src.io import ensure_dirs, save_table, update_metrics, read_metrics
ensure_dirs()

In [2]:
from src.io import load_raw
from src.preprocessing import clean_frame, apply_funnel, length_band

raw = load_raw()
print(f"raw rows: {len(raw):,}   columns: {list(raw.columns)}")
raw.head(3)

raw rows: 74,412   columns: ['review_id', 'app_name', 'platform', 'review_text', 'language', 'star_rating', 'review_date', 'clean_text', 'satisfaction_label', 'aspect_tags']


,review_id,app_name,platform,review_text,language,star_rating,review_date,clean_text,satisfaction_label,aspect_tags
0,72c1ab3aff4f1d79,UAE PASS,Google Play,جيد,Arabic,5,2026-07-18,جيد,Satisfied,NaN
1,682f594dc22630af,UAE PASS,Google Play,جيده,Arabic,3,2026-07-17,جيده,Neutral,NaN
2,4a7efa56616aadbb,UAE PASS,Google Play,ممتاز,Arabic,5,2026-07-16,ممتاز,Satisfied,NaN


## Recompute derived columns

The scraper wrote `language`, `clean_text` and `satisfaction_label` at collection time. I recompute them here so that this notebook, not the scraper, is the single source of truth for the rules.

In [3]:
df = clean_frame(raw)

# how often does the recomputed language / label disagree with the scraper's version?
lang_changed = (df["language"] != raw["language"]).sum()
print(f"language recomputed differently for {lang_changed:,} rows "
      f"({lang_changed/len(df):.2%}) - expected to be small, the rules are the same")
print()
print(df["language"].value_counts())
print()
print(df["satisfaction_label"].fillna("Neutral / missing").value_counts())

language recomputed differently for 1 rows (0.00%) - expected to be small, the rules are the same

language
English    54215
Arabic     19731
mixed        398
unknown       68
Name: count, dtype: int64

satisfaction_label
Satisfied            37838
Dissatisfied         32443
Neutral / missing     4131
Name: count, dtype: int64


## Before / after examples of the normalisation

In [4]:
ex = df[(df.language == "Arabic") & (df.word_count >= 6)].sample(3, random_state=config.RANDOM_SEED)
for _, r in ex.iterrows():
    print("RAW  :", r.review_text[:140])
    print("CLEAN:", r.clean_text[:140])
    print()
ex = df[(df.language == "English") & (df.review_text.str.contains("http|!!!|ooo", regex=True))].head(2)
for _, r in ex.iterrows():
    print("RAW  :", r.review_text[:140])
    print("CLEAN:", r.clean_text[:140])
    print()

RAW  : انتظرت يوم كامل و لم تصل الشحنة تطبيق فاشل و شركة كسولة و ضعيفة
CLEAN: انتظرت يوم كامل و لم تصل الشحنه تطبيق فاشل و شركه كسوله و ضعيفه

RAW  : انا من العراق وهذي السفرة الخامسة لي في دبي حقيقآ برنامج رائع واستفدت منه كثير اثناء القيادة . طبعآ في شوارع مامحدثة واسماء المطاعم  ، بالتو
CLEAN: انا من العراق وهذي السفره الخامسه لي في دبي حقيقا برنامج رايع واستفدت منه كثير اثناء القياده طبعا في شوارع مامحدثه واسماء المطاعم ، بالتوفيق

RAW  : البرنامج لا يعمل بشكل جيد والدفع عند SMS فقط
CLEAN: البرنامج لا يعمل بشكل جيد والدفع عند sms فقط

RAW  : goood
CLEAN: good

RAW  : So gooooooood
CLEAN: so good



## Inclusion funnel

In [5]:
modelling, funnel = apply_funnel(df)
funnel.to_csv(config.FUNNEL_FILE, index=False)
save_table(funnel, "cleaning_funnel")
funnel

,step,rows_before,rows_after,removed,retention_pct
0,Remove empty review text,74412,74411,1,100.0
1,Remove exact duplicate reviews,74411,74248,163,99.8
2,Keep substantive reviews (>= 6 words),74248,52640,21608,70.7
3,Remove neutral band (3-star) and missing stars,52640,49217,3423,66.1
4,Restrict to Arabic or English (drop mixed / ot...,49217,48930,287,65.8


In [6]:
print(f"modelling set: {len(modelling):,} rows "
      f"({len(modelling)/len(df):.1%} of raw)")
print()
print(pd.crosstab(modelling.language, modelling.satisfaction_label, margins=True))
print()
print((pd.crosstab(modelling.language, modelling.satisfaction_label, normalize="index") * 100).round(1))

modelling set: 48,930 rows (65.8% of raw)



satisfaction_label  Dissatisfied  Satisfied    All
language                                          
Arabic                      5045       6824  11869
English                    22145      14916  37061
All                        27190      21740  48930

satisfaction_label  Dissatisfied  Satisfied
language                                   
Arabic                      42.5       57.5
English                     59.8       40.2


## Save

The full cleaned frame (all rows, with a `kept` flag) goes to `data/interim/`; the modelling set goes to `data/processed/`.

In [7]:
df["kept"] = df["review_id"].isin(modelling["review_id"])
df["length_band"] = length_band(df["word_count"])
modelling["length_band"] = length_band(modelling["word_count"])

config.INTERIM_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(config.CLEAN_FILE, index=False)
modelling.to_parquet(config.MODELLING_FILE, index=False)

update_metrics("dataset", {
    "raw_rows": int(len(raw)),
    "apps": int(raw["app_name"].nunique()),
    "modelling_rows": int(len(modelling)),
    "funnel": funnel.to_dict(orient="records"),
    "language_counts_raw": df["language"].value_counts().to_dict(),
    "language_counts_modelling": modelling["language"].value_counts().to_dict(),
    "label_by_language": pd.crosstab(modelling.language, modelling.satisfaction_label).to_dict(),
    "date_min": str(df.review_date.min().date()), "date_max": str(df.review_date.max().date()),
})
print("saved", config.CLEAN_FILE.name, "and", config.MODELLING_FILE.name)

saved reviews_clean.parquet and modelling_set.parquet
